In [0]:
# ---- Config ----
CATALOG = "workspace"
BRONZE_SCHEMA = "careerpulse_bronze"
BRONZE_TABLE  = f"{CATALOG}.{BRONZE_SCHEMA}.job_postings_raw"

BASE_URL = "https://www.themuse.com/api/public/jobs"
MAX_PAGES = 50  # keep small while testing
REQUEST_PARAMS = {}  # no key needed


In [0]:
import requests, json, time
from datetime import datetime, timezone, timedelta
from pyspark.sql import Row

def fetch_page(page: int, params: dict) -> dict:
    p = dict(params)
    p["page"] = page
    r = requests.get(BASE_URL, params=p, timeout=30)
    r.raise_for_status()
    return r.json()

ingestion_ts = datetime.now(timezone.utc)
ingestion_date = ingestion_ts.date().isoformat()

# Get the last ingestion date from Bronze
last_run = spark.sql(f"""
    SELECT MAX(ingestion_date) FROM {BRONZE_TABLE}
""").collect()[0][0]

# Default to yesterday if no prior run exists
if last_run is None:
    cutoff = (datetime.now(timezone.utc) - timedelta(days=1)).date().isoformat()
else:
    cutoff = last_run

# Loop through pages
rows = []
for page in range(1, MAX_PAGES + 1):
    payload = fetch_page(page, REQUEST_PARAMS)
    results = payload.get("results", [])

    rows.append(Row(
        ingestion_ts=ingestion_ts.isoformat(),
        ingestion_date=ingestion_date,
        page=page,
        request_url=BASE_URL,
        request_params=json.dumps(REQUEST_PARAMS),
        payload_json=json.dumps(payload),
    ))
    time.sleep(0.5)

    # Check AFTER appending whether we've gone far enough back
    if all(r["publication_date"][:10] < cutoff for r in results):
        break

if not rows:
    dbutils.notebook.exit("No new postings since last ingestion — skipping write.")

df = spark.createDataFrame(rows)

(df.write
  .format("delta")
  .mode("append")
  .saveAsTable(BRONZE_TABLE))


In [0]:
# Fetch first page to get total page count, then loop
first_page = fetch_page(1, REQUEST_PARAMS)
first_page["page_count"]

In [0]:
%sql
--verify there is data in BRONZE_TABLE
SELECT COUNT(*) AS n_rows
FROM workspace.careerpulse_bronze.job_postings_raw;

In [0]:
%sql
--look at top 10 rows to verify table is populating
SELECT ingestion_ts, ingestion_date, page, request_url
FROM workspace.careerpulse_bronze.job_postings_raw
ORDER BY ingestion_ts DESC, page DESC
LIMIT 10;

In [0]:
%sql
--inspect the payload results in json format
SELECT payload_json
FROM workspace.careerpulse_bronze.job_postings_raw
ORDER BY ingestion_ts DESC, page DESC
LIMIT 1;
